### PART1.引入

训练神经网络时，最常用的算法就是**反向传播**。

在该算法中，模型参数（模型权重）会根据损失函数对各个参数的梯度来更新调整。为计算这些梯度，PyTorch 内置了一套微分引擎：torch.autograd。它可以对任意计算图自动求解梯度。

举一个最简单的单层神经网络例子：输入为x，参数为权重w、偏置b，再搭配某个损失函数。在 PyTorch 中可以按如下方式定义该网络。

In [2]:
import torch

x = torch.ones(5)  # 输入的数据
y = torch.zeros(3)  # 期望的数据（标签）
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
#损失函数为交叉熵损失函数

在这个网络中，w 和 b 是待优化的模型参数。因此我们需要计算损失函数对这些变量的梯度。为实现该功能，我们需要为这些张量开启 requires_grad 属性。

你可以在创建张量时就设置requires_grad，也可以后续调用x.requires_grad_(True)方法开启。

我们作用在张量上、用来构建计算图的运算，实际上是Function类的对象。该对象既知道如何执行前向计算，也清楚在反向传播阶段如何求导。反向传播函数的引用保存在张量的grad_fn属性中。你可以查阅文档获取更多关于Function的信息。

In [3]:
print(f"z 的梯度函数 = {z.grad_fn}")
print(f"loss 的梯度函数 = {loss.grad_fn}")

z 的梯度函数 = <AddBackward0 object at 0x0000023BBD6F20E0>
loss 的梯度函数 = <BinaryCrossEntropyWithLogitsBackward0 object at 0x0000023BBE2C37F0>


### PART2.计算梯度

想要优化神经网络的参数权重，我们需要计算损失函数对各个参数的导数。也就是在输入x、标签y固定时，求出 $\displaystyle \frac{\partial loss}{\partial w}$ 和 $\displaystyle \frac{\partial loss}{\partial b}$。
想要算出这些导数，调用loss.backward()，之后就可以从w.grad和b.grad读取梯度值。

In [4]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0511, 0.0186, 0.1666],
        [0.0511, 0.0186, 0.1666],
        [0.0511, 0.0186, 0.1666],
        [0.0511, 0.0186, 0.1666],
        [0.0511, 0.0186, 0.1666]])
tensor([0.0511, 0.0186, 0.1666])


### PART3.关闭梯度追踪

默认情况下，所有设置requires_grad=True的张量都会记录计算历史，支持梯度求解。但有些场景我们并不需要求梯度。例如：模型训练完成后，仅对输入数据做推理，也就是只执行网络的前向计算。

以下情况您可能需要关闭梯度追踪：
- 将神经网络中部分参数设置为冻结参数（不参与训练更新）。
- 只执行前向传播时，加快计算速度；不对张量做梯度追踪，计算效率会更高。

我们可以用torch.no_grad()代码块包裹计算逻辑，以此关闭计算追踪。

In [5]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


还有另一种方式可以实现相同效果：对张量调用detach()方法。

In [6]:
# 矩阵乘法+偏置，w、b开启梯度，因此z会开启梯度追踪
z = torch.matmul(x, w)+b
# 打印z是否需要计算梯度
print(z.requires_grad)

# with torch.no_grad()：进入「禁用自动求导」上下文环境
with torch.no_grad():
    # 同样的运算，但是不会构建计算图，不追踪梯度
    z = torch.matmul(x, w)+b
# 此时z不再需要梯度
print(z.requires_grad)

True
False


### PART4.深入理解计算图

从概念上讲，autograd 将数据（张量）以及全部执行过的运算（包括运算产出的新张量），记录在一张由Function对象构成的有向无环图 (DAG)中。

在这张 DAG 里：叶子节点是输入张量，根节点是输出张量。从根节点向叶子节点遍历这张图，就可以依靠链式法则自动计算梯度。

前向传播过程中，autograd 会同时完成两件事：
- 执行运算，计算得到输出张量
- 在 DAG 图中保存该运算对应的梯度函数

当对 DAG 的根节点调用.backward()，就会触发反向传播，autograd 执行如下操作：
- 通过每一个grad_fn计算梯度
- 将梯度累加存入对应张量的.grad属性
- 使用链式法则，一直反向传播，传递到各个叶子张量

PyTorch 中的 DAG 是**动态**的

有一个重要知识点：每次调用.backward()之后，计算图都会从零重新构建，autograd 会开始生成一张全新的图。
正是动态图这个特性，让你可以在模型内部使用控制流语句；如果需要，每一轮迭代都可以改变张量形状、大小以及执行的运算。

PyTorch 动态图，每一轮前向传播才当场搭计算图，支持 if、for 循环；对比 TensorFlow1.x 是静态图，要先把整个图定义好再跑。

### 附：张量梯度与雅可比乘积

大多数场景下，我们得到标量损失，然后求损失对各参数的梯度。
但也存在输出是任意形状张量的情况。此时 PyTorch 不会直接求出完整梯度，而是计算所谓雅可比乘积（Jacobian product）。

对于向量函数 $\vec y = f(\vec x)$，其中 $\vec x=\langle x_1,\dots,x_n\rangle$，$\vec y=\langle y_1,\dots,y_m\rangle$。
$\vec y$ 对 $\vec x$ 的梯度由雅可比矩阵给出：

$$
J=
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \dots & \frac{\partial y_1}{\partial x_n}\\
\vdots & \ddots & \vdots\\
\frac{\partial y_m}{\partial x_1} & \dots & \frac{\partial y_m}{\partial x_n}
\end{pmatrix}
$$

PyTorch并不会直接计算完整的雅可比矩阵，而是对于给定输入向量 $v=(v_1 \dots v_m)$，计算**雅可比乘积 $v^T \cdot J$**。
实现方式：调用`backward(v)`，将向量$v$作为传入参数。
$v$ 的形状，必须和待求导的原始输出张量的形状保持一致。

In [9]:
# 创建4行5列的单位矩阵张量，开启梯度计算
inp = torch.eye(4, 5, requires_grad=True)
# 每个元素+1，再平方，最后转置；得到输出张量out
out = (inp+1).pow(2).t()

# 第一次反向传播，传入和out形状一致的全1向量，保留计算图不释放
out.backward(torch.ones_like(out), retain_graph=True)
print(f"第一次反向传播\n{inp.grad}")

# 第二次反向传播，再次求导；grad会累加，不会自动清零
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\n第二次\n{inp.grad}")

# 手动把inp的梯度全部置0
inp.grad.zero_()

# 清零之后再做一次反向传播
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\n清零后反向传播\n{inp.grad}")

第一次反向传播
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

第二次
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

清零后反向传播
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
